# Stage 2 and Stage 3 Run Commands

Current workflow for this repo:

- SKU-110K data is in `data/SKU110K`.
- Product detector checkpoint is `s3_iou_resnet50_csv_06.h5`.
- Stage 2 writes `data/processed/s3_detections.json`.
- Stage 3 combines `row_predictions.csv` and `detections.json`.

Run the H5 detector workflow first. Fallback options are at the bottom.

In [38]:
import os
from pathlib import Path

PROJECT_ROOT = Path('/Users/dipali/Documents/Coursework/ACV/Final_projectt/CS543_row_detection')
os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

Working directory: /Users/dipali/Documents/Coursework/ACV/Final_projectt/CS543_row_detection


## 0. Check Required Files

In [39]:
paths = [
    Path('s3_iou_resnet50_csv_06.h5'),
    Path('data/SKU110K/images'),
    Path('data/SKU110K/annotations/annotations_train.csv'),
    Path('data/SKU110K/annotations/annotations_val.csv'),
    Path('data/processed/SHARD/images'),
    Path('data/processed/s3_row_predictions.csv'),
]

for path in paths:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

s3_iou_resnet50_csv_06.h5: OK
data/SKU110K/images: OK
data/SKU110K/annotations/annotations_train.csv: OK
data/SKU110K/annotations/annotations_val.csv: OK
data/processed/SHARD/images: OK
data/processed/s3_row_predictions.csv: OK


## 1. Environment For Provided H5 Detector

`s3_iou_resnet50_csv_06.h5` is an old Keras RetinaNet model. It usually requires a Python 3.10 environment with `keras_retinanet`. Python 3.13/Keras 3 may fail.

Run these commands in a terminal if this notebook kernel cannot import `keras_retinanet`.

In [17]:
# Terminal commands:
# conda create -n sku-retinanet python=3.10 -y
# conda activate sku-retinanet
# python -m pip install tensorflow==2.13.1 keras==2.13.1 opencv-python pillow tqdm h5py matplotlib
# python pip install tensorflow-metal
# python -m pip install git+https://github.com/fizyr/keras-retinanet.git

In [40]:
import importlib.util
print('keras_retinanet installed:', importlib.util.find_spec('keras_retinanet') is not None)

keras_retinanet installed: True


In [41]:
import sys
print(sys.executable)

/Users/dipali/miniconda3/envs/sku-retinanet/bin/python


In [42]:
import tensorflow as tf
print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Stage 1: Export Shelf Row Predictions

Skip if `data/processed/s3_row_predictions.csv` already exists and is current.

In [43]:
!python scripts/s3_export_row_predictions.py \
  --checkpoint runs/row_dht_1d/s3_best.pt \
  --images_dir data/processed/SHARD/images \
  --output data/processed/s3_row_predictions.csv \
  --threshold 0.35 \
  --min_distance 20

Traceback (most recent call last):
  File "/Users/dipali/Documents/Coursework/ACV/Final_projectt/CS543_row_detection/scripts/s3_export_row_predictions.py", line 11, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'


## 3. Stage 2: Product Detection With Provided H5 Model

This runs the SKU-110K-pretrained Keras RetinaNet detector on the SHARD shelf images and writes the actual Stage 3 input: `data/processed/s3_detections.json`.

In [44]:
!python scripts/s3_predict_product_detections_h5.py \
  --model s3_iou_resnet50_csv_06.h5 \
  --images_dir data/processed/SHARD/images \
  --output data/processed/s3_detections.json \
  --score_threshold 0.5 \
  --max_detections 100000000

Resuming: 6836 images already processed, skipping.
Loading model: s3_iou_resnet50_csv_06.h5
2026-05-13 04:00:00.381148: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-05-13 04:00:00.381171: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-13 04:00:00.381176: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-05-13 04:00:00.381758: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-13 04:00:00.381776: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
Images total: 6000 | Remaining: 0
Detecting products: 0it [00:00, ?it/s]
Wrote 446952 detectio

## 4. Visualize Stage 2 Detections

Check whether detector boxes look reasonable before localization.

In [45]:
!python scripts/s3_visualize_detections.py \
  --detections data/processed/s3_detections.json \
  --image_dir data/processed/SHARD/images \
  --output_dir runs/product_detection_visualizations \
  --limit 50

SKIP missing image: data/processed/SHARD/images/00022940-426d-43a0-b100-517aa34b8a56.jpg
SKIP missing image: data/processed/SHARD/images/00049bc9-1001-46db-9ec9-2d339c9c7da7.jpg
SKIP missing image: data/processed/SHARD/images/00085d8f-b6e2-4f35-a803-eb3e1317512c.jpg
SKIP missing image: data/processed/SHARD/images/00124552-9786-4f7e-889f-f21b3f120924.jpg
SKIP missing image: data/processed/SHARD/images/00255bfa-f19e-4912-bd8d-64292be0ee29.jpg
SKIP missing image: data/processed/SHARD/images/0025f55f-7980-42c1-a4ee-d5b8b1a73e05.jpg
SKIP missing image: data/processed/SHARD/images/0029d24c-f645-4ae2-9a53-0def5d82eede.jpg
SKIP missing image: data/processed/SHARD/images/0031e040-c0b0-4a70-8beb-8504b764e380.jpg
SKIP missing image: data/processed/SHARD/images/0032c764-8f84-4ee2-af28-94823c464723.jpg
SKIP missing image: data/processed/SHARD/images/00465ec1-47c7-46c7-bbeb-ce8505b841c8.jpg
SKIP missing image: data/processed/SHARD/images/0047e4f2-b1d6-44f3-ba92-3a54c4435154.jpg
SKIP missing image: d

## 5. Stage 3: Product Localization

Assign each detected product to shelf row, column, and subrow.

In [46]:
!python scripts/s3_localize.py \
  --row_preds data/processed/s3_row_predictions.csv \
  --detections data/processed/s3_detections.json \
  --image_dir data/processed/SHARD/images \
  --output data/processed/s3_localized_products.json \
  --output_csv data/processed/s3_localized_products.csv

Loading row predictions …
Loading product detections …
  SKIP 00022940-426d-43a0-b100-517aa34b8a56.jpg: Image file not found: data/processed/SHARD/images/00022940-426d-43a0-b100-517aa34b8a56.jpg
  SKIP 00049bc9-1001-46db-9ec9-2d339c9c7da7.jpg: Image file not found: data/processed/SHARD/images/00049bc9-1001-46db-9ec9-2d339c9c7da7.jpg
  SKIP 00085d8f-b6e2-4f35-a803-eb3e1317512c.jpg: Image file not found: data/processed/SHARD/images/00085d8f-b6e2-4f35-a803-eb3e1317512c.jpg
  SKIP 00124552-9786-4f7e-889f-f21b3f120924.jpg: Image file not found: data/processed/SHARD/images/00124552-9786-4f7e-889f-f21b3f120924.jpg
  SKIP 00255bfa-f19e-4912-bd8d-64292be0ee29.jpg: Image file not found: data/processed/SHARD/images/00255bfa-f19e-4912-bd8d-64292be0ee29.jpg
  SKIP 0025f55f-7980-42c1-a4ee-d5b8b1a73e05.jpg: Image file not found: data/processed/SHARD/images/0025f55f-7980-42c1-a4ee-d5b8b1a73e05.jpg
  SKIP 0029d24c-f645-4ae2-9a53-0def5d82eede.jpg: Image file not found: data/processed/SHARD/images/0029d2

## 6. Visualize Stage 3 Localization

In [47]:
!python scripts/s3_visualize_localization.py \
  --localized data/processed/s3_localized_products.json \
  --image_dir data/processed/SHARD/images \
  --row_preds data/processed/s3_row_predictions.csv \
  --output_dir runs/localization_visualizations \
  --limit 50

Wrote 50 visualizations to: runs/localization_visualizations


## 7. Inspect Outputs

In [48]:
import json

outputs = [
    Path('data/processed/s3_detections.json'),
    Path('data/processed/s3_localized_products.json'),
    Path('data/processed/s3_localized_products.csv'),
    Path('runs/product_detection_visualizations'),
    Path('runs/localization_visualizations'),
]

for path in outputs:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

det_path = Path('data/processed/s3_detections.json')
if det_path.exists():
    detections = json.loads(det_path.read_text())
    print('detections:', len(detections))
    print('sample:', detections[0] if detections else None)

data/processed/s3_detections.json: OK
data/processed/s3_localized_products.json: OK
data/processed/s3_localized_products.csv: OK
runs/product_detection_visualizations: OK
runs/localization_visualizations: OK
detections: 446952
sample: {'image_name': '00022940-426d-43a0-b100-517aa34b8a56.jpg', 'x1': 112.97, 'y1': 91.05, 'x2': 183.35, 'y2': 130.69, 'score': 0.95993, 'class_id': 0, 'category_id': 1, 'source': 'keras_retinanet_h5'}
